In [7]:
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent

In [15]:
llm = ChatGroq(model='openai/gpt-oss-120b')

In [16]:
@tool
def add(a: int, b: int):
    """Add two numbers."""
    return a + b



In [17]:
# list of exisitng tools
tools = [add]
llm_with_tools = llm.bind_tools(tools)

In [18]:
# Agent node
def call_model(state: MessagesState) -> dict:
    response = llm.invoke(
        state["messages"]
    )
    return {
        "messages": [response]
    }

In [19]:
# Router
def should_continue(state: MessagesState)-> str:
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tools"
    return END

In [20]:
# Router
def should_continue(state: MessagesState)-> str:
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tools"
    return END
# Build Graph-
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools)) # without creating def tools, this node was created via  langgraph.prebuilt's ToolNode.

# Add edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {
    "tools": "tools",
    END: END
})
builder.add_edge("tools", "agent")


# Add checkointer

checkpointer = InMemorySaver()
graph = builder.compile(
    checkpointer = checkpointer
)

In [21]:
# Thread A

config_a = {
    "configurable": {
        "thread_id": "thread-A"
    }
}


# Thread B

config_b = {
    "configurable": {
        "thread_id": "thread-B"
    }
}


In [22]:
# Tell Thread A something

graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Sam."
            }
        ]
    },
    config_a
)


# Tell Thread B something different

graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Alex."
            }
        ]
    },
    config_b
)

{'messages': [HumanMessage(content='My name is Alex.', additional_kwargs={}, response_metadata={}, id='dd1801fc-a1a2-4424-98e9-70271c852a66'),
  AIMessage(content='Nice to meet you, Alex! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "My name is Alex." Likely they are introducing themselves. We should respond politely, maybe ask how can we help. There\'s no disallowed content. So we can respond with greeting.'}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 76, 'total_tokens': 141, 'completion_time': 0.140241472, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.002888749, 'prompt_tokens_details': None, 'queue_time': 0.285596087, 'total_time': 0.143130221}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_88d246c915', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f6555-ff4b-7322-85f2-c4484d509e88-0', tool_calls=[], i

In [24]:
# Ask Thread B

result_a = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config_a
)

In [25]:
# Ask Thread B

result_b = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config_b
)


print(
    "Thread A:",
    result_a["messages"][-1].content
)

print(
    "Thread B:",
    result_b["messages"][-1].content
)

Thread A: Your name is Sam.
Thread B: Your name is Alex.
